# Phase 1: Data Inspection and Cleaning

In this phase, we load the four datasets, inspect their structure, combine the two web activity files and keep only clients included in the experiment.

In [1]:
# Import libraries

import pandas as pd
import numpy as np

# Load the four datasets

df_demo = pd.read_csv("data/df_final_demo.txt")
df_experiment = pd.read_csv("data/df_final_experiment_clients.txt")
df_web_1 = pd.read_csv(
    "data/df_final_web_data_pt_1.zip",
    compression="zip"
)
df_web_2 = pd.read_csv(
    "data/df_final_web_data_pt_2.zip",
    compression="zip"
)

# Confirm that the files loaded correctly

print("Client profiles:", df_demo.shape)
print("Experiment roster:", df_experiment.shape)
print("Web data part 1:", df_web_1.shape)
print("Web data part 2:", df_web_2.shape)

Client profiles: (70609, 9)
Experiment roster: (70609, 2)
Web data part 1: (343141, 5)
Web data part 2: (412264, 5)


## 1. Initial Dataset Inspection

We inspect the shape, columns, data types, missing values and duplicate rows in each dataset.

In [2]:
# Store the datasets in a dictionary for quick inspection

datasets = {
    "Client Profiles": df_demo,
    "Experiment Roster": df_experiment,
    "Web Data Part 1": df_web_1,
    "Web Data Part 2": df_web_2
}

# Inspect each dataset

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print("\nData types:")
    print(df.dtypes)
    print("\nMissing values:")
    print(df.isna().sum())
    print("\nDuplicate rows:", df.duplicated().sum())
    display(df.head())


Client Profiles
--------------------------------------------------
Shape: (70609, 9)
Columns: ['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']

Data types:
client_id             int64
clnt_tenure_yr      float64
clnt_tenure_mnth    float64
clnt_age            float64
gendr                object
num_accts           float64
bal                 float64
calls_6_mnth        float64
logons_6_mnth       float64
dtype: object

Missing values:
client_id            0
clnt_tenure_yr      14
clnt_tenure_mnth    14
clnt_age            15
gendr               14
num_accts           14
bal                 14
calls_6_mnth        14
logons_6_mnth       14
dtype: int64

Duplicate rows: 0


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0



Experiment Roster
--------------------------------------------------
Shape: (70609, 2)
Columns: ['client_id', 'Variation']

Data types:
client_id     int64
Variation    object
dtype: object

Missing values:
client_id        0
Variation    20109
dtype: int64

Duplicate rows: 0


,client_id,Variation
0,9988021,Test
1,8320017,Test
2,4033851,Control
3,1982004,Test
4,9294070,Control



Web Data Part 1
--------------------------------------------------
Shape: (343141, 5)
Columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']

Data types:
client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
dtype: object

Missing values:
client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

Duplicate rows: 2095


,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04



Web Data Part 2
--------------------------------------------------
Shape: (412264, 5)
Columns: ['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time']

Data types:
client_id        int64
visitor_id      object
visit_id        object
process_step    object
date_time       object
dtype: object

Missing values:
client_id       0
visitor_id      0
visit_id        0
process_step    0
date_time       0
dtype: int64

Duplicate rows: 8669


,client_id,visitor_id,visit_id,process_step,date_time
0,763412,601952081_10457207388,397475557_40440946728_419634,confirm,2017-06-06 08:56:00
1,6019349,442094451_91531546617,154620534_35331068705_522317,confirm,2017-06-01 11:59:27
2,6019349,442094451_91531546617,154620534_35331068705_522317,step_3,2017-06-01 11:58:48
3,6019349,442094451_91531546617,154620534_35331068705_522317,step_2,2017-06-01 11:58:08
4,6019349,442094451_91531546617,154620534_35331068705_522317,step_1,2017-06-01 11:57:58


## 2. Combine the Web Activity Files

The two web files contain the same columns and different activity rows. We stack them vertically using `concat`.

In [3]:
# Combine both parts of the web activity dataset

df_web = pd.concat(
    [df_web_1, df_web_2],
    ignore_index=True
)

# Validate the result

print("Rows in part 1:", len(df_web_1))
print("Rows in part 2:", len(df_web_2))
print("Expected total:", len(df_web_1) + len(df_web_2))
print("Combined total:", len(df_web))

display(df_web.head())

Rows in part 1: 343141
Rows in part 2: 412264
Expected total: 755405
Combined total: 755405


,client_id,visitor_id,visit_id,process_step,date_time
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04


## 3. Check Duplicate Web Records

The same client may appear many times because each row represents an activity. We only remove rows that are completely identical.

In [4]:
# Count exact duplicate rows

print("Exact duplicate rows:", df_web.duplicated().sum())

# Remove exact duplicates

df_web = df_web.drop_duplicates().reset_index(drop=True)

print("Web data shape after duplicate removal:", df_web.shape)

Exact duplicate rows: 10764
Web data shape after duplicate removal: (744641, 5)



## 4. Validate the Experiment Time Window

The A/B test ran from March 15, 2017 to June 20, 2017. We convert the web activity timestamps to datetime format and verify that the recorded activity falls within the official experiment period.

In [5]:
# Convert the timestamp column to datetime format

df_web["date_time"] = pd.to_datetime(
    df_web["date_time"],
    errors="coerce"
)

# Define the official experiment period

experiment_start = pd.Timestamp("2017-03-15")
experiment_end = pd.Timestamp("2017-06-20 23:59:59")

# Check the observed date range

print("First activity:", df_web["date_time"].min())
print("Last activity:", df_web["date_time"].max())

# Identify activity outside the experiment period

outside_experiment = df_web[
    (df_web["date_time"] < experiment_start) |
    (df_web["date_time"] > experiment_end)
]

print("\nActivities outside experiment period:", len(outside_experiment))

First activity: 2017-03-15 00:03:03
Last activity: 2017-06-20 23:59:57

Activities outside experiment period: 0


## 5. Merge Client Profiles with Experiment Assignment

We combine the client profile dataset with the experiment roster using `client_id`.

The experiment roster contains the `Variation` column, which identifies whether a client belongs to the Test group, the Control group or has no assigned variation. At this stage, we keep all matched clients so we can inspect the experiment assignments before filtering.

In [6]:
# Merge client profiles with the experiment roster

df_clients = df_demo.merge(
    df_experiment,
    on="client_id",
    how="inner",
    validate="one_to_one"
)

# Inspect the merged client-level dataset

print("Clients in profile data:", df_demo["client_id"].nunique())
print("Clients in experiment roster:", df_experiment["client_id"].nunique())
print("Clients after merge:", df_clients["client_id"].nunique())

print("\nExperiment assignments:")
print(df_clients["Variation"].value_counts(dropna=False))

display(df_clients.head())

Clients in profile data: 70609
Clients in experiment roster: 70609
Clients after merge: 70609

Experiment assignments:
Variation
Test       26968
Control    23532
NaN        20109
Name: count, dtype: int64


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,Variation
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,Test
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,Control
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,Test
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,Test
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,Control


## 6. Keep Valid Experiment Participants

We merge the web activity with the experiment roster to attach the Test or Control assignment to every activity row.

Clients without a valid Test or Control assignment are excluded because they were not part of either experimental group.

In [7]:
# Add the experiment assignment to each web activity row

df_web_experiment = df_web.merge(
    df_experiment,
    on="client_id",
    how="inner",
    validate="many_to_one"
)

# Inspect Test, Control and missing assignments before filtering

print("Assignments before filtering:")
print(df_web_experiment["Variation"].value_counts(dropna=False))

# Keep only valid Test and Control participants

df_web_experiment = (
    df_web_experiment[
        df_web_experiment["Variation"].isin(["Test", "Control"])
    ]
    .reset_index(drop=True)
)

# Validate the filtered experiment data

print("\nExperiment activity rows:", len(df_web_experiment))
print("Experiment clients:", df_web_experiment["client_id"].nunique())

print("\nClients per group:")
print(
    df_web_experiment.groupby("Variation")["client_id"]
    .nunique()
)

print("\nActivity rows per group:")
print(df_web_experiment["Variation"].value_counts())

Assignments before filtering:
Variation
Test       176699
Control    140536
NaN        126662
Name: count, dtype: int64

Experiment activity rows: 317235
Experiment clients: 50500

Clients per group:
Variation
Control    23532
Test       26968
Name: client_id, dtype: int64

Activity rows per group:
Variation
Test       176699
Control    140536
Name: count, dtype: int64


## 7. Create the Final Analysis Dataset

We merge the filtered experiment web activity with the client profile dataset.

The final dataset contains:

- client demographics
- account information
- Test or Control assignment
- website activity
- process steps
- timestamps

In [8]:
# Add client profile information to each experiment activity row

df_final = df_web_experiment.merge(
    df_demo,
    on="client_id",
    how="left",
    validate="many_to_one"
)

# Standardize column names

df_final.columns = (
    df_final.columns
    .str.strip()
    .str.lower()
)

# Validate the final dataset

print("Final dataset shape:", df_final.shape)
print("Unique clients:", df_final["client_id"].nunique())
print("Unique visits:", df_final["visit_id"].nunique())

print("\nColumns:")
print(df_final.columns.tolist())

display(df_final.head())


Final dataset shape: (317235, 14)
Unique clients: 50500
Unique visits: 69205

Columns:
['client_id', 'visitor_id', 'visit_id', 'process_step', 'date_time', 'variation', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth']


,client_id,visitor_id,visit_id,process_step,date_time,variation,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth
0,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:27:07,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
1,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:26:51,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
2,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:19:22,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
3,9988021,580560515_7732621733,781255054_21935453173_531117,step_2,2017-04-17 15:19:13,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0
4,9988021,580560515_7732621733,781255054_21935453173_531117,step_3,2017-04-17 15:18:04,Test,5.0,64.0,79.0,U,2.0,189023.86,1.0,4.0


In [9]:
print(df_web_experiment["Variation"].value_counts(dropna=False))


Variation
Test       176699
Control    140536
Name: count, dtype: int64


## 8. Final Data Quality Checks

We check the final dataset for duplicate rows, missing values, valid experiment groups, expected process steps and the correct experiment period before calculating performance metrics.

In [10]:
# Check exact duplicate rows

print("Duplicate rows:", df_final.duplicated().sum())

# Check missing values

print("\nMissing values:")
print(
    df_final.isna()
    .sum()
    .sort_values(ascending=False)
)

# Check unique clients per experiment group

print("\nUnique clients per group:")
print(
    df_final.groupby("variation")["client_id"]
    .nunique()
)

# Check expected process steps

print("\nProcess steps:")
print(
    df_final["process_step"]
    .value_counts(dropna=False)
)

# Check the final activity period

print("\nActivity period:")
print("First activity:", df_final["date_time"].min())
print("Last activity:", df_final["date_time"].max())

Duplicate rows: 0

Missing values:
clnt_age            112
clnt_tenure_yr      100
clnt_tenure_mnth    100
gendr               100
num_accts           100
bal                 100
calls_6_mnth        100
logons_6_mnth       100
client_id             0
visitor_id            0
visit_id              0
process_step          0
date_time             0
variation             0
dtype: int64

Unique clients per group:
variation
Control    23532
Test       26968
Name: client_id, dtype: int64

Process steps:
process_step
start      101153
step_1      68210
step_2      56672
step_3      48264
confirm     42936
Name: count, dtype: int64

Activity period:
First activity: 2017-03-15 00:19:28
Last activity: 2017-06-20 23:57:06


## 9. Missing Data Assessment

We examined the remaining missing values after merging the datasets.

Only 13 out of 50,500 experiment clients (0.03%) have incomplete demographic information. Since this represents a very small proportion of the data, these clients are retained for behavioural analyses and excluded only when calculating demographic statistics.

In [11]:
# Identify rows with missing client profile information

missing_profile = df_final[
    df_final[
        [
            "clnt_age",
            "clnt_tenure_yr",
            "clnt_tenure_mnth",
            "gendr",
            "num_accts",
            "bal",
            "calls_6_mnth",
            "logons_6_mnth"
        ]
    ].isna().any(axis=1)
]

print("Rows with missing profile data:", len(missing_profile))
print(
    "Unique clients with missing profile data:",
    missing_profile["client_id"].nunique()
)

display(
    missing_profile[
        [
            "client_id",
            "variation",
            "clnt_age",
            "clnt_tenure_yr",
            "gendr"
        ]
    ].drop_duplicates()
)

Rows with missing profile data: 112
Unique clients with missing profile data: 13


,client_id,variation,clnt_age,clnt_tenure_yr,gendr
24845,8191345,Control,NaN,NaN,NaN
34070,5144725,Test,NaN,NaN,NaN
42693,7616759,Control,NaN,NaN,NaN
89564,2222915,Test,NaN,NaN,NaN
92760,5277910,Test,NaN,NaN,NaN
113082,8412164,Test,NaN,NaN,NaN
172616,355337,Control,NaN,NaN,NaN
183335,4666211,Control,NaN,8.0,F
183559,1227228,Test,NaN,NaN,NaN
228672,1037867,Test,NaN,NaN,NaN


## 10. Phase 1 Summary

The datasets were successfully loaded, inspected and merged into a single analysis dataset.

Key outcomes:

- Combined the two web activity datasets.
- Attached the experiment assignment (Test or Control).
- Merged client demographic information.
- Verified the experiment period.
- Removed clients without a valid experiment assignment.
- Confirmed there are no duplicate rows.
- Identified only 13 clients with incomplete demographic data.

The final dataset is now ready for performance metric analysis.

## Phase 2: Performance Metrics

This notebook continues the analysis completed in Phase 1.

The objective is to compare the performance of Vanguard’s redesigned interface with the traditional interface using descriptive KPIs and to explore where differences occur within the user journey.

The analysis covers:

- **KPI 1:** Completion rate
- **Additional KPI:** Step-level conversion and drop-off
- **KPI 2:** Time spent between process steps
- **KPI 3:** Backward-transition rate (error proxy)

The additional funnel analysis examines where clients progress or drop off within the process and provides more actionable insights for potential UX improvements and future A/B tests.

The notebook also includes **Hypothesis 3**, which statistically tests whether step-level conversion differs between the Test and Control groups.

The required overall completion-rate hypotheses are handled separately in the main experiment evaluation.


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from statsmodels.stats.proportion import proportions_ztest

In [13]:
# Exported from Notebook 1 after Phase 1 cleaning.
experiment_final_df = pd.read_csv("experiment_final_df.csv")

print("Shape:", experiment_final_df.shape)
print("Unique clients:", experiment_final_df["client_id"].nunique())

experiment_final_df.head()

Shape: (50487, 12)
Unique clients: 50487


,client_id,clnt_tenure_yr,clnt_tenure_mnth,clnt_age,gendr,num_accts,bal,calls_6_mnth,logons_6_mnth,tenure_check_correct,Variation,furthest_step_reached
0,836976,6.0,73.0,60.5,U,2.0,45105.30,6.0,9.0,6.0,Test,confirm
1,2304905,7.0,94.0,58.0,U,2.0,110860.30,6.0,9.0,7.0,Control,confirm
2,1439522,5.0,64.0,32.0,U,2.0,52467.79,6.0,9.0,5.0,Test,step_3
3,1562045,16.0,198.0,49.0,M,2.0,67454.65,3.0,6.0,16.0,Test,start
4,5126305,12.0,145.0,33.0,F,2.0,103671.75,0.0,3.0,12.0,Control,start


### Validate available columns

Before calculating completion rate, we confirm the exact column names in the client-level dataframe.

This prevents errors caused by using a column name that does not exist or has different capitalization.

In [14]:
# Display all column names in the Phase 1 client-level dataframe
print(experiment_final_df.columns.tolist())

['client_id', 'clnt_tenure_yr', 'clnt_tenure_mnth', 'clnt_age', 'gendr', 'num_accts', 'bal', 'calls_6_mnth', 'logons_6_mnth', 'tenure_check_correct', 'Variation', 'furthest_step_reached']


## KPI 1 – Completion Rate

Completion rate is calculated at client level.

A client is classified as completed when `furthest_step_reached` equals `confirm`.

This ensures each client is counted once, regardless of how many visits or web events they generated.

In [15]:
# Create a client-level completion indicator
client_completion_df = experiment_final_df[
    ["client_id", "Variation", "furthest_step_reached"]
].copy()

client_completion_df["completed"] = (
    client_completion_df["furthest_step_reached"] == "confirm"
)

client_completion_df.head()

,client_id,Variation,furthest_step_reached,completed
0,836976,Test,confirm,True
1,2304905,Control,confirm,True
2,1439522,Test,step_3,False
3,1562045,Test,start,False
4,5126305,Control,start,False


### Completion indicator check

The `completed` column correctly identifies whether each client reached the `confirm` step.

Clients with `confirm` are marked `True`. Clients whose furthest step was `start`, `step_1`, `step_2` or `step_3` are marked `False`.

In [16]:
# Summarize completion by experiment group
completion_summary = (
    client_completion_df
    .groupby("Variation")
    .agg(
        total_clients=("client_id", "nunique"),
        completed_clients=("completed", "sum")
    )
)

completion_summary["completion_rate"] = (
    completion_summary["completed_clients"]
    / completion_summary["total_clients"]
)

completion_summary["completion_rate_percent"] = (
    completion_summary["completion_rate"] * 100
)

completion_summary

,total_clients,completed_clients,completion_rate,completion_rate_percent
Variation,,,,
Control,23526,15428,0.655785,65.578509
Test,26961,18682,0.692927,69.292682


In [17]:
# Calculate the practical improvement of Test over Control

test_completion_rate = completion_summary.loc["Test", "completion_rate"]
control_completion_rate = completion_summary.loc["Control", "completion_rate"]

absolute_difference = test_completion_rate - control_completion_rate
relative_improvement = absolute_difference / control_completion_rate

print(f"Control completion rate: {control_completion_rate:.2%}")
print(f"Test completion rate: {test_completion_rate:.2%}")
print(f"Absolute difference: {absolute_difference:.2%}")
print(f"Relative improvement: {relative_improvement:.2%}")

Control completion rate: 65.58%
Test completion rate: 69.29%
Absolute difference: 3.71%
Relative improvement: 5.66%


### Completion-rate interpretation

The Test group achieved a completion rate of 69.29%, compared with 65.58% for the Control group.

This is:

- an absolute improvement of 3.71 percentage points
- a relative improvement of 5.66%

The observed relative improvement exceeds Vanguard’s 5% cost-effectiveness threshold. Statistical testing is still required to determine whether this difference is unlikely to be due to chance.

## Additional KPI – Funnel Reach and Step-to-Step Drop-Off

Overall completion rate tells us whether clients reached the final `confirm` step, but it does not show where users stop progressing through the journey.

To make the analysis more actionable, we also calculate:

- how many clients reached each process step
- the conversion rate from one step to the next
- the drop-off rate between consecutive steps
- differences between the Test and Control groups

This helps identify which part of the journey contributes most to abandonment and which stage could be prioritized in a future A/B test or UX improvement.

In [18]:
# Define the expected order of the funnel

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

funnel_client_df = experiment_final_df[
    ["client_id", "Variation", "furthest_step_reached"]
].copy()

funnel_client_df["furthest_step_number"] = (
    funnel_client_df["furthest_step_reached"].map(step_order)
)

funnel_client_df.head()

,client_id,Variation,furthest_step_reached,furthest_step_number
0,836976,Test,confirm,4
1,2304905,Control,confirm,4
2,1439522,Test,step_3,3
3,1562045,Test,start,0
4,5126305,Control,start,0


### Calculate funnel reach

A client is counted as having reached a step when their furthest recorded step is equal to or beyond that stage.

This creates a cumulative funnel for the Test and Control groups and shows how many clients remain at each stage of the process.

In [19]:
# Calculate the number and percentage of clients reaching each funnel stage

funnel_rows = []

for variation in ["Control", "Test"]:
    group = funnel_client_df[
        funnel_client_df["Variation"] == variation
    ]

    total_clients = group["client_id"].nunique()

    for step_name, step_number in step_order.items():
        clients_reached = (
            group["furthest_step_number"] >= step_number
        ).sum()

        funnel_rows.append({
            "Variation": variation,
            "process_step": step_name,
            "clients_reached": clients_reached,
            "reach_rate": clients_reached / total_clients
        })

funnel_summary = pd.DataFrame(funnel_rows)

funnel_summary

,Variation,process_step,clients_reached,reach_rate
0,Control,start,23526,1.000000
1,Control,step_1,20241,0.860367
2,Control,step_2,18786,0.798521
3,Control,step_3,17521,0.744750
4,Control,confirm,15428,0.655785
5,Test,start,26961,1.000000
6,Test,step_1,24507,0.908980
7,Test,step_2,22528,0.835577
8,Test,step_3,21118,0.783280
9,Test,confirm,18682,0.692927


### Calculate step-to-step conversion and drop-off

To understand where users discontinue the process, we calculate:

- the conversion rate between consecutive steps
- the corresponding drop-off rate

This provides more detailed insight than the overall completion rate and helps identify which stage could benefit most from future interface improvements.

In [20]:
# Calculate step-to-step conversion and drop-off

funnel_summary["previous_clients_reached"] = (
    funnel_summary
    .groupby("Variation")["clients_reached"]
    .shift(1)
)

funnel_summary["step_conversion_rate"] = (
    funnel_summary["clients_reached"]
    / funnel_summary["previous_clients_reached"]
)

funnel_summary["drop_off_rate"] = (
    1 - funnel_summary["step_conversion_rate"]
)

funnel_summary

,Variation,process_step,clients_reached,reach_rate,previous_clients_reached,step_conversion_rate,drop_off_rate
0,Control,start,23526,1.000000,NaN,NaN,NaN
1,Control,step_1,20241,0.860367,23526.0,0.860367,0.139633
2,Control,step_2,18786,0.798521,20241.0,0.928116,0.071884
3,Control,step_3,17521,0.744750,18786.0,0.932663,0.067337
4,Control,confirm,15428,0.655785,17521.0,0.880543,0.119457
5,Test,start,26961,1.000000,NaN,NaN,NaN
6,Test,step_1,24507,0.908980,26961.0,0.908980,0.091020
7,Test,step_2,22528,0.835577,24507.0,0.919248,0.080752
8,Test,step_3,21118,0.783280,22528.0,0.937411,0.062589
9,Test,confirm,18682,0.692927,21118.0,0.884648,0.115352


### Compare funnel performance between Test and Control

To make the funnel easier to interpret, we compare the step-level conversion and drop-off rates for both experiment groups side by side.

This highlights at which stage the redesigned interface performs better or worse than the original interface.

In [21]:
# Compare conversion and drop-off rates side by side

funnel_comparison = funnel_summary.pivot(
    index="process_step",
    columns="Variation",
    values=["step_conversion_rate", "drop_off_rate"]
)

funnel_comparison

step_conversion_rate           drop_off_rate          
Variation                 Control      Test       Control      Test
process_step                                                       
confirm                  0.880543  0.884648      0.119457  0.115352
start                         NaN       NaN           NaN       NaN
step_1                   0.860367  0.908980      0.139633  0.091020
step_2                   0.928116  0.919248      0.071884  0.080752
step_3                   0.932663  0.937411      0.067337  0.062589

### Compare differences between Test and Control

To identify where the redesign had the greatest impact, we calculate the difference in conversion and drop-off rates between the Test and Control groups for each funnel step.

In [22]:
# Create a clean comparison table

funnel_difference = pd.DataFrame({
    "Control Conversion": funnel_comparison["step_conversion_rate"]["Control"],
    "Test Conversion": funnel_comparison["step_conversion_rate"]["Test"],
    "Conversion Difference": (
        funnel_comparison["step_conversion_rate"]["Test"]
        - funnel_comparison["step_conversion_rate"]["Control"]
    ),
    "Control Drop-off": funnel_comparison["drop_off_rate"]["Control"],
    "Test Drop-off": funnel_comparison["drop_off_rate"]["Test"],
    "Drop-off Difference": (
        funnel_comparison["drop_off_rate"]["Test"]
        - funnel_comparison["drop_off_rate"]["Control"]
    )
})

funnel_difference

,Control Conversion,Test Conversion,Conversion Difference,Control Drop-off,Test Drop-off,Drop-off Difference
process_step,,,,,,
confirm,0.880543,0.884648,0.004105,0.119457,0.115352,-0.004105
start,NaN,NaN,NaN,NaN,NaN,NaN
step_1,0.860367,0.908980,0.048612,0.139633,0.091020,-0.048612
step_2,0.928116,0.919248,-0.008869,0.071884,0.080752,0.008869
step_3,0.932663,0.937411,0.004749,0.067337,0.062589,-0.004749


In [23]:
# Add clear transition labels for interpretation

transition_labels = {
    "step_1": "start → step_1",
    "step_2": "step_1 → step_2",
    "step_3": "step_2 → step_3",
    "confirm": "step_3 → confirm"
}

funnel_difference = funnel_difference.drop(index="start").copy()

funnel_difference["transition"] = (
    funnel_difference.index.map(transition_labels)
)

funnel_difference

,Control Conversion,Test Conversion,Conversion Difference,Control Drop-off,Test Drop-off,Drop-off Difference,transition
process_step,,,,,,,
confirm,0.880543,0.884648,0.004105,0.119457,0.115352,-0.004105,step_3 → confirm
step_1,0.860367,0.908980,0.048612,0.139633,0.091020,-0.048612,start → step_1
step_2,0.928116,0.919248,-0.008869,0.071884,0.080752,0.008869,step_1 → step_2
step_3,0.932663,0.937411,0.004749,0.067337,0.062589,-0.004749,step_2 → step_3


### Additional KPI Interpretation – Step-Level Conversion and Drop-Off

The funnel analysis shows that the Test group has a higher overall completion rate, but the redesign does not improve every transition equally.

The largest improvement occurs from `start → step_1`:

- Control conversion: 86.04%
- Test conversion: 90.90%
- Improvement: approximately 4.86 percentage points
- Test drop-off is approximately 4.86 percentage points lower

For `step_1 → step_2`, however, the Test group performs slightly worse:

- Control conversion: 92.81%
- Test conversion: 91.92%
- Test drop-off is approximately 0.89 percentage points higher

The remaining differences are smaller:

- `step_2 → step_3`: Test conversion is approximately 0.47 percentage points higher
- `step_3 → confirm`: Test conversion is approximately 0.41 percentage points higher

Descriptively, the largest contribution to the Test group's higher overall completion rate appears to occur at the beginning of the journey, while `step_1 → step_2` is the only transition where the redesigned experience shows higher drop-off.

These differences have not yet been statistically tested. The `step_1 → step_2` transition could therefore be investigated further as a potential focus for a follow-up A/B test or UX improvement.

## KPI 2 – Time Spent on Each Step

The client-level dataframe cannot be used for time analysis because it contains one row per client.

For this KPI, we use the original web activity data. Each row represents one event. Events are sorted chronologically within each `visit_id`.

Time spent on a step is estimated as the difference between the timestamp of the current event and the timestamp of the next event within the same visit.

In [24]:
# Load the two web activity files and the experiment assignment file

web_data_pt1_df = pd.read_csv(
    "data/df_final_web_data_pt_1.zip",
    compression="zip"
)

web_data_pt2_df = pd.read_csv(
    "data/df_final_web_data_pt_2.zip",
    compression="zip"
)

experiment_clients_df = pd.read_csv(
    "data/df_final_experiment_clients.txt"
)

print("Web part 1:", web_data_pt1_df.shape)
print("Web part 2:", web_data_pt2_df.shape)
print("Experiment clients:", experiment_clients_df.shape)

Web part 1: (343141, 5)
Web part 2: (412264, 5)
Experiment clients: (70609, 2)


### Create the event-level experiment dataset

The two web activity files are combined into a single event-level dataset.

This dataset is then merged with the experiment assignment file so that every event is labelled as either **Test** or **Control**.

Finally, the events are sorted chronologically within each visit, which is required to calculate the time between consecutive steps.

In [25]:
# Combine both web activity datasets
combined_web_data_df = pd.concat(
    [web_data_pt1_df, web_data_pt2_df],
    ignore_index=True
).drop_duplicates()

# Convert timestamps
combined_web_data_df["date_time"] = pd.to_datetime(
    combined_web_data_df["date_time"]
)

# Merge with experiment assignment
event_level_df = combined_web_data_df.merge(
    experiment_clients_df,
    on="client_id",
    how="inner"
)

# Keep only Test and Control
event_level_df = event_level_df[
    event_level_df["Variation"].isin(["Test", "Control"])
].copy()

# Sort events chronologically
event_level_df = event_level_df.sort_values(
    ["visit_id", "date_time"]
).reset_index(drop=True)

In [26]:
# Validate the event-level dataset

print("Rows:", len(event_level_df))
print("Unique clients:", event_level_df["client_id"].nunique())
print("Unique visits:", event_level_df["visit_id"].nunique())

print("\nVariation counts:")
print(event_level_df["Variation"].value_counts())

event_level_df.head()

Rows: 317235
Unique clients: 50500
Unique visits: 69205

Variation counts:
Variation
Test       176699
Control    140536
Name: count, dtype: int64


,client_id,visitor_id,visit_id,process_step,date_time,Variation
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,Test
1,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:23:09,Test
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test


### Calculate time between consecutive events

The time spent on a process step is estimated as the time difference between one event and the next event within the same visit.

The last event of a visit has no following event and is therefore excluded from the calculation.

In [27]:
# Calculate the next event within each visit

event_level_df["next_time"] = (
    event_level_df.groupby("visit_id")["date_time"].shift(-1)
)

event_level_df["next_step"] = (
    event_level_df.groupby("visit_id")["process_step"].shift(-1)
)

# Calculate elapsed time in seconds

event_level_df["time_seconds"] = (
    event_level_df["next_time"] - event_level_df["date_time"]
).dt.total_seconds()

# Remove the final event of each visit
time_df = event_level_df.dropna(subset=["time_seconds"]).copy()

# Remove negative durations if they exist
time_df = time_df[time_df["time_seconds"] >= 0]

time_df.head()

,client_id,visitor_id,visit_id,process_step,date_time,Variation,next_time,next_step,time_seconds
0,3561384,451664975_1722933822,100012776_37918976071_457913,confirm,2017-04-26 13:22:17,Test,2017-04-26 13:23:09,confirm,52.0
2,7338123,612065484_94198474375,100019538_17884295066_43909,start,2017-04-09 16:20:56,Test,2017-04-09 16:21:12,step_1,16.0
3,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:12,Test,2017-04-09 16:21:21,step_2,9.0
4,7338123,612065484_94198474375,100019538_17884295066_43909,step_2,2017-04-09 16:21:21,Test,2017-04-09 16:21:35,step_1,14.0
5,7338123,612065484_94198474375,100019538_17884295066_43909,step_1,2017-04-09 16:21:35,Test,2017-04-09 16:21:41,step_1,6.0


In [28]:
# Validate calculated durations

print("Transitions:", len(time_df))

print("\nSummary statistics:")
print(time_df["time_seconds"].describe())

Transitions: 248030

Summary statistics:
count    248030.000000
mean         84.207939
std         215.986776
min           0.000000
25%          13.000000
50%          36.000000
75%          83.000000
max       40235.000000
Name: time_seconds, dtype: float64


### Inspect extreme duration values before filtering

Before excluding any observations, we inspect unusually long gaps between consecutive events.

Durations above 30 minutes may represent inactivity or users leaving the process open rather than continuous interaction. The 30-minute threshold is an analytical assumption, so the extreme observations are reviewed before applying the filter.

In [29]:
# Inspect event gaps longer than 30 minutes

time_outliers_df = time_df[
    time_df["time_seconds"] > 1800
].copy()

print("Total valid transitions:", len(time_df))
print("Transitions > 30 minutes:", len(time_outliers_df))

print(
    f"Percentage > 30 minutes: "
    f"{len(time_outliers_df) / len(time_df) * 100:.2f}%"
)

time_outliers_df[
    [
        "client_id",
        "visit_id",
        "Variation",
        "process_step",
        "next_step",
        "time_seconds"
    ]
].sort_values(
    "time_seconds",
    ascending=False
).head(20)

Total valid transitions: 248030
Transitions > 30 minutes: 354
Percentage > 30 minutes: 0.14%


,client_id,visit_id,Variation,process_step,next_step,time_seconds
258400,2182225,831987489_84761163210_335684,Control,start,start,40235.0
132852,6392043,478417687_30092205462_548423,Test,start,start,24819.0
134725,9553121,483738263_4057680487_61869,Control,step_2,start,21763.0
101869,4647786,391696103_92230204739_479887,Test,confirm,confirm,14581.0
176525,8883167,602386199_13818409466_144534,Test,confirm,start,12980.0
185467,6420672,627596734_90499955288_891366,Control,start,start,11207.0
140846,9971962,501042989_50588313479_700675,Test,step_2,start,11204.0
171796,5282553,588904389_16165570985_286826,Control,step_1,start,10286.0
177527,2882702,605153028_51517094625_529507,Control,confirm,confirm,9396.0
110676,4803107,416635057_22840074620_580752,Test,step_1,start,8797.0


### Handling extreme time values

The duration inspection identified 354 transitions above 30 minutes, representing only 0.14% of all valid transitions.

Although these observations are rare, some gaps last several hours, with a maximum above 11 hours. These values are unlikely to represent continuous active interaction and may disproportionately affect the average.

For the primary time-spent analysis, durations above 30 minutes are excluded. The unfiltered data is retained separately, and median duration is also reported because it is less sensitive to extreme values.

In [30]:
# Remove unrealistic inactive sessions (>30 minutes)

time_filtered_df = time_df[
    time_df["time_seconds"] <= 1800
].copy()

print("Transitions before filtering:", len(time_df))
print("Transitions after filtering:", len(time_filtered_df))

time_filtered_df["time_seconds"].describe()

Transitions before filtering: 248030
Transitions after filtering: 247676


count    247676.000000
mean         79.987912
std         147.247367
min           0.000000
25%          13.000000
50%          36.000000
75%          82.000000
max        1800.000000
Name: time_seconds, dtype: float64

### Average and median time by process step

After excluding event gaps above 30 minutes, the average and median duration are calculated for each process step and experiment group.

The median is particularly important because time data remains right-skewed even after filtering.

In [31]:
# Summarize time spent by experiment group and process step

time_summary = (
    time_filtered_df
    .groupby(["Variation", "process_step"])
    .agg(
        transition_count=("time_seconds", "count"),
        average_seconds=("time_seconds", "mean"),
        median_seconds=("time_seconds", "median")
    )
    .reset_index()
)

time_summary["average_minutes"] = (
    time_summary["average_seconds"] / 60
)

time_summary["median_minutes"] = (
    time_summary["median_seconds"] / 60
)

time_summary

,Variation,process_step,transition_count,average_seconds,median_seconds,average_minutes,median_minutes
0,Control,confirm,1996,156.160321,49.0,2.602672,0.816667
1,Control,start,35676,60.911733,20.0,1.015196,0.333333
2,Control,step_1,26026,47.528433,20.0,0.792141,0.333333
3,Control,step_2,24305,90.214112,64.0,1.503569,1.066667
4,Control,step_3,20251,133.298306,72.0,2.221638,1.200000
5,Test,confirm,4172,221.972915,94.0,3.699549,1.566667
6,Test,start,46241,56.240912,14.0,0.937349,0.233333
7,Test,step_1,35500,58.481380,27.0,0.974690,0.450000
8,Test,step_2,29571,88.133949,61.0,1.468899,1.016667
9,Test,step_3,23938,124.832693,57.0,2.080545,0.950000



### Review the impact of the `confirm` step

The `confirm` step is the final stage of the process and has no required subsequent step.

Before deciding whether it should be included in the primary time-spent KPI, we compare the results with and without `confirm`.

This sensitivity check shows whether post-confirmation activity materially affects the overall time comparison between the Test and Control groups.

In [32]:
# Compare overall time results with and without the confirm step

with_confirm = (
    time_filtered_df
    .groupby("Variation")["time_seconds"]
    .agg(
        average_with_confirm="mean",
        median_with_confirm="median"
    )
)

without_confirm = (
    time_filtered_df[
        time_filtered_df["process_step"] != "confirm"
    ]
    .groupby("Variation")["time_seconds"]
    .agg(
        average_without_confirm="mean",
        median_without_confirm="median"
    )
)

confirm_impact = with_confirm.join(without_confirm)

confirm_impact["average_difference"] = (
    confirm_impact["average_without_confirm"]
    - confirm_impact["average_with_confirm"]
)

confirm_impact["median_difference"] = (
    confirm_impact["median_without_confirm"]
    - confirm_impact["median_with_confirm"]
)

confirm_impact.round(2)

,average_with_confirm,median_with_confirm,average_without_confirm,median_without_confirm,average_difference,median_difference
Variation,,,,,,
Control,79.57,37.0,78.13,37.0,-1.44,0.0
Test,80.31,35.0,75.94,34.0,-4.37,-1.0


### Sensitivity Check Interpretation

Including `confirm` affects the Test group more strongly than the Control group.

- Control mean: 79.57 seconds with `confirm` vs 78.13 seconds without it.
- Test mean: 80.31 seconds with `confirm` vs 75.94 seconds without it.
- Median times change very little.

With `confirm` included, the Test group appears slightly slower overall. After excluding `confirm`, the Test group has a lower overall mean time.

Because `confirm` is the final stage and has no required next step, these post-confirmation durations do not represent progression through the funnel.

### KPI 2 Data-Quality Adjustment

Based on the sensitivity check, `confirm` is excluded from the primary step-time KPI.

Durations starting from `confirm` represent activity after the user has already reached the final process stage, such as repeated confirmation events or later navigation.

The primary KPI therefore focuses on:

`start`, `step_1`, `step_2` and `step_3`

The `confirm` results are retained separately as a sensitivity check rather than discarded from the analysis.

In [33]:
# Create the final KPI 2 summary excluding confirm

time_kpi_df = time_summary[
    time_summary["process_step"] != "confirm"
].copy()

time_kpi_df

,Variation,process_step,transition_count,average_seconds,median_seconds,average_minutes,median_minutes
1,Control,start,35676,60.911733,20.0,1.015196,0.333333
2,Control,step_1,26026,47.528433,20.0,0.792141,0.333333
3,Control,step_2,24305,90.214112,64.0,1.503569,1.066667
4,Control,step_3,20251,133.298306,72.0,2.221638,1.200000
6,Test,start,46241,56.240912,14.0,0.937349,0.233333
7,Test,step_1,35500,58.481380,27.0,0.974690,0.450000
8,Test,step_2,29571,88.133949,61.0,1.468899,1.016667
9,Test,step_3,23938,124.832693,57.0,2.080545,0.950000


In [34]:
# Compare average and median time by process step

time_comparison = time_kpi_df.pivot(
    index="process_step",
    columns="Variation",
    values=["average_seconds", "median_seconds"]
)

time_comparison

average_seconds             median_seconds      
Variation            Control        Test        Control  Test
process_step                                                 
start              60.911733   56.240912           20.0  14.0
step_1             47.528433   58.481380           20.0  27.0
step_2             90.214112   88.133949           64.0  61.0
step_3            133.298306  124.832693           72.0  57.0

### KPI 2 Interpretation

The descriptive results show that the Test group does not have shorter interaction times at every process step.

Compared with the Control group:

- Test users progressed faster after `start`.
- Test users spent more time after reaching `step_1`.
- Test users progressed slightly faster after `step_2`.
- Test users progressed noticeably faster after `step_3`.

The median times show the same overall pattern and are considered the more robust measure because the duration distribution is right-skewed.

The sensitivity check also shows that including post-`confirm` activity affects the overall mean, particularly for the Test group. With `confirm` included, Test appears slightly slower overall (80.31 vs 79.57 seconds). After excluding `confirm`, Test has a lower overall mean time (75.94 vs 78.13 seconds).

Because `confirm` is the final process stage and has no required next step, it is excluded from the primary step-time KPI. Post-confirmation timing is retained separately as a sensitivity check.

Overall, the Test group shows shorter interaction times at most stages, while `step_1` shows the opposite pattern.

These results are descriptive and have not been statistically tested, so they should not be interpreted as confirmed effects of the redesigned interface.

## KPI 3 – Backward-Transition Rate (Error Proxy)

A backward transition is treated as a possible error or indicator of user friction, rather than a confirmed error.

The expected process order is:

`start → step_1 → step_2 → step_3 → confirm`

A transition is classified as backward when the next recorded step has a lower position than the current step within the same visit.

For example:

`step_2 → step_1`

is classified as a backward transition.

Because users may intentionally return to review or change information, backward navigation is used as an error proxy rather than interpreted as a confirmed user error.

The backward-transition rate is calculated as:

\[
\text{Backward-Transition Rate} =
\frac{\text{Number of backward transitions}}
{\text{Total valid transitions}}
\]

A lower backward-transition rate indicates less revisiting of earlier steps, although the reason for backward navigation cannot be determined from the available data.

In [35]:
# Assign a numeric order to each process step

step_order = {
    "start": 0,
    "step_1": 1,
    "step_2": 2,
    "step_3": 3,
    "confirm": 4
}

time_df["current_step_number"] = (
    time_df["process_step"].map(step_order)
)

time_df["next_step_number"] = (
    time_df["next_step"].map(step_order)
)

time_df["backward_transition"] = (
    time_df["next_step_number"]
    < time_df["current_step_number"]
)

time_df[
    [
        "visit_id",
        "Variation",
        "process_step",
        "next_step",
        "backward_transition"
    ]
].head(10)

,visit_id,Variation,process_step,next_step,backward_transition
0,100012776_37918976071_457913,Test,confirm,confirm,False
2,100019538_17884295066_43909,Test,start,step_1,False
3,100019538_17884295066_43909,Test,step_1,step_2,False
4,100019538_17884295066_43909,Test,step_2,step_1,True
5,100019538_17884295066_43909,Test,step_1,step_1,False
6,100019538_17884295066_43909,Test,step_1,start,True
7,100019538_17884295066_43909,Test,start,start,False
8,100019538_17884295066_43909,Test,start,step_1,False
9,100019538_17884295066_43909,Test,step_1,step_2,False
10,100019538_17884295066_43909,Test,step_2,step_3,False


In [36]:
# Calculate backward-transition rate for Test and Control

backward_summary = (
    time_df
    .groupby("Variation")
    .agg(
        total_transitions=("backward_transition", "count"),
        backward_transitions=("backward_transition", "sum")
    )
)

backward_summary["backward_transition_rate"] = (
    backward_summary["backward_transitions"]
    / backward_summary["total_transitions"]
)

backward_summary["backward_transition_rate_percent"] = (
    backward_summary["backward_transition_rate"] * 100
)

backward_summary

,total_transitions,backward_transitions,backward_transition_rate,backward_transition_rate_percent
Variation,,,,
Control,108402,9682,0.089316,8.931570
Test,139628,16358,0.117154,11.715415


### Compare backward-transition rates

The backward-transition rates of the Test and Control groups are compared using both the absolute difference and the relative difference.

This shows whether the redesigned interface is associated with more or less backward navigation than the original interface.

Because backward navigation is used as an error proxy, the comparison should be interpreted as a difference in revisiting behaviour or potential friction rather than confirmed navigation errors.

In [37]:
# Compare Test and Control backward-transition rates

test_backward_rate = backward_summary.loc[
    "Test",
    "backward_transition_rate"
]

control_backward_rate = backward_summary.loc[
    "Control",
    "backward_transition_rate"
]

absolute_difference = (
    test_backward_rate - control_backward_rate
)

relative_difference = (
    absolute_difference / control_backward_rate
)

print(
    f"Control backward-transition rate: "
    f"{control_backward_rate:.2%}"
)

print(
    f"Test backward-transition rate: "
    f"{test_backward_rate:.2%}"
)

print(
    f"Absolute difference: "
    f"{absolute_difference:.2%}"
)

print(
    f"Relative difference: "
    f"{relative_difference:.2%}"
)

Control backward-transition rate: 8.93%
Test backward-transition rate: 11.72%
Absolute difference: 2.78%
Relative difference: 31.17%


### KPI 3 Interpretation

The Test group shows a higher backward-transition rate than the Control group.

Compared with the Control group:

- Control backward-transition rate: 8.93%
- Test backward-transition rate: 11.72%

This represents an absolute increase of approximately 2.78 percentage points.

Descriptively, the Test experience is associated with more frequent backward navigation, while also achieving a higher overall completion rate.

Backward navigation may reflect users reviewing information, correcting an earlier choice or experiencing friction. The available event data does not allow us to determine the underlying reason.

The higher backward-transition rate should therefore be considered alongside the higher completion rate when evaluating the redesigned experience.

These differences are descriptive and have not yet been statistically tested.

# Phase 2 Summary

The descriptive KPI analysis identifies several differences between the Test and Control groups:

- **Overall completion:** Test achieved a higher completion rate (69.29%) than Control (65.58%), a difference of 3.71 percentage points.
- **Funnel progression:** The largest observed difference occurred at `start → step_1`, where Test conversion was approximately 4.86 percentage points higher.
- **Drop-off:** `step_1 → step_2` was the only transition where Test showed higher drop-off, by approximately 0.89 percentage points.
- **Time spent:** After excluding post-`confirm` activity, Test showed shorter median interaction times at most stages, while `step_1` showed the opposite pattern. The sensitivity check demonstrated that including `confirm` particularly affects the Test group's overall mean.
- **Backward transitions:** Test had a higher backward-transition rate (11.72%) than Control (8.93%).

Overall, the descriptive results show a mixed pattern. Test has higher completion and stronger progression through most of the funnel, but also more backward navigation and slightly weaker progression from `step_1 → step_2`.

These findings describe observed differences only. Phase 3 hypothesis testing is required to determine which differences are statistically significant.

# Phase 3 – Hypothesis Testing

Phase 2 identified observed differences between the Test and Control groups. Phase 3 evaluates whether these differences are statistically significant and whether the redesign meets Vanguard's business requirement.

A significance level of **α = 0.05** is used throughout the analysis.

The analysis includes:

1. Completion rate difference between Test and Control
2. Vanguard's 5% cost-effectiveness threshold
3. Step-level funnel conversion differences
4. Demographic balance between Test and Control

## Hypothesis 1 – Completion Rate Difference

### Research Question

Is the difference in completion probability between the Test and Control groups statistically significant?

In Phase 2, completion rate was calculated as the proportion of clients who reached the final `confirm` step. This proportion can also be interpreted as the estimated probability that a client completes the process.

For the Test group:

\[
P(\text{Completion} \mid \text{Test}) =
\frac{18,682}{26,961}
= 0.6929
= 69.29\%
\]

For the Control group:

\[
P(\text{Completion} \mid \text{Control}) =
\frac{15,428}{23,526}
= 0.6558
= 65.58\%
\]

The observed difference in completion probability is therefore approximately **3.71 percentage points**.

Hypothesis testing is used to determine whether this observed difference is statistically significant or could reasonably be explained by sampling variation.

### Hypotheses

**Null hypothesis (H₀):**

\[
H_0: p_{Test} = p_{Control}
\]

The probability of completion is the same for the Test and Control groups.

**Alternative hypothesis (H₁):**

\[
H_1: p_{Test} \neq p_{Control}
\]

The probability of completion differs between the Test and Control groups.

Because completion is a binary outcome and we are comparing proportions from two independent groups, a **two-proportion z-test** is appropriate.

Significance level:

\[
\alpha = 0.05
\]

In [45]:
from statsmodels.stats.proportion import proportions_ztest
import numpy as np

# Retrieve completion counts calculated in Phase 2

test_completed = int(
    completion_summary.loc["Test", "completed_clients"]
)

test_total = int(
    completion_summary.loc["Test", "total_clients"]
)

control_completed = int(
    completion_summary.loc["Control", "completed_clients"]
)

control_total = int(
    completion_summary.loc["Control", "total_clients"]
)

print("Test completed:", test_completed)
print("Test total:", test_total)
print("Control completed:", control_completed)
print("Control total:", control_total)

Test completed: 18682
Test total: 26961
Control completed: 15428
Control total: 23526


In [43]:
# Two-sided two-proportion z-test

counts = np.array([
    test_completed,
    control_completed
])

nobs = np.array([
    test_total,
    control_total
])

z_stat_h1, p_value_h1 = proportions_ztest(
    counts,
    nobs,
    alternative="two-sided"
)

alpha = 0.05

print(f"Z-statistic: {z_stat_h1:.4f}")
print(f"P-value: {p_value_h1:.6g}")

if p_value_h1 < alpha:
    print("Reject H0: completion rates are statistically significantly different.")
else:
    print("Fail to reject H0: there is insufficient evidence of a difference.")

Z-statistic: 8.8927
P-value: 5.96164e-19
Reject H0: completion rates are statistically significantly different.


In [48]:
# 95% confidence interval for the difference in completion rates

from statsmodels.stats.proportion import confint_proportions_2indep

test_rate = test_completed / test_total
control_rate = control_completed / control_total

observed_difference = test_rate - control_rate

ci_low, ci_high = confint_proportions_2indep(
    count1=test_completed,
    nobs1=test_total,
    count2=control_completed,
    nobs2=control_total,
    method="wald"
)

print(f"Test completion rate: {test_rate:.2%}")
print(f"Control completion rate: {control_rate:.2%}")
print(f"Observed difference: {observed_difference:.2%}")
print(f"95% Confidence Interval: ({ci_low:.2%}, {ci_high:.2%})")

Test completion rate: 69.29%
Control completion rate: 65.58%
Observed difference: 3.71%
95% Confidence Interval: (2.89%, 4.53%)


### Hypothesis 1 Conclusion

The Test group had a completion probability of **69.29%**, compared with **65.58%** for the Control group, an observed increase of **3.71 percentage points**.

The two-proportion z-test produced a **p-value below 0.05**, so we reject the null hypothesis.

The **95% confidence interval for the difference in completion rates is approximately 2.89% to 4.53%**. Since the entire interval is above zero, this provides further evidence that the higher completion rate in the Test group is not explained by random sampling variation.

**Conclusion:** The redesigned interface produced a statistically significant improvement in completion probability. Based on the 95% confidence interval, the true absolute improvement is estimated to be approximately **2.89 to 4.53 percentage points**.

## Hypothesis 2 – 5% Cost-Effectiveness Threshold

Vanguard requires the redesigned interface to increase the completion rate by at least **5% relative to the Control completion rate** to justify the costs of implementation.

From Phase 2:

- Control completion probability = **65.58%**
- Test completion probability = **69.29%**
- Observed relative improvement = approximately **5.66%**

Although the observed uplift is above 5%, hypothesis testing is required to determine whether there is sufficient statistical evidence that the true improvement exceeds Vanguard's 5% threshold.

### Hypotheses

**Null hypothesis (H₀):**

\[
H_0: p_{Test} \leq 1.05 \times p_{Control}
\]

The redesigned interface does not improve completion by more than 5% relative to Control.

**Alternative hypothesis (H₁):**

\[
H_1: p_{Test} > 1.05 \times p_{Control}
\]

The redesigned interface improves completion by more than 5% relative to Control.

A **one-sided two-proportion z-test** is used with:

\[
\alpha = 0.05
\]

In [50]:
# Calculate Vanguard's 5% relative uplift threshold

test_rate = test_completed / test_total
control_rate = control_completed / control_total

required_test_rate = control_rate * 1.05

observed_relative_improvement = (
    test_rate - control_rate
) / control_rate

print(f"Control completion rate: {control_rate:.2%}")
print(f"Test completion rate: {test_rate:.2%}")
print(f"Required Test rate for +5% uplift: {required_test_rate:.2%}")
print(f"Observed relative improvement: {observed_relative_improvement:.2%}")

Control completion rate: 65.58%
Test completion rate: 69.29%
Required Test rate for +5% uplift: 68.86%
Observed relative improvement: 5.66%


In [51]:
# One-sided two-proportion z-test for the 5% relative uplift threshold

from statsmodels.stats.proportion import proportions_ztest
import numpy as np

counts = np.array([
    test_completed,
    control_completed
])

nobs = np.array([
    test_total,
    control_total
])

# A 5% relative uplift over Control corresponds to this required difference
required_difference = control_rate * 0.05

z_stat_h2, p_value_h2 = proportions_ztest(
    counts,
    nobs,
    value=required_difference,
    alternative="larger"
)

print(f"Required absolute difference: {required_difference:.2%}")
print(f"Observed absolute difference: {(test_rate - control_rate):.2%}")
print(f"Z-statistic: {z_stat_h2:.4f}")
print(f"P-value: {p_value_h2:.6f}")

Required absolute difference: 3.28%
Observed absolute difference: 3.71%
Z-statistic: 1.0421
P-value: 0.148682


### Hypothesis 2 Conclusion

The Test group achieved an observed relative improvement of approximately **5.66%** over Control, slightly exceeding Vanguard's required **5% cost-effectiveness threshold**.

However, the one-sided hypothesis test produced a **z-statistic of 1.0421** and a **p-value of 0.1487**.

Since the p-value is greater than the significance level of 0.05, we **fail to reject the null hypothesis**.

Although the observed sample performance exceeds the 5% threshold, there is insufficient statistical evidence to conclude that the true improvement exceeds 5%.

**Conclusion:** The redesign shows promising improvement, but Vanguard's 5% cost-effectiveness requirement is **not statistically confirmed**.

## Hypothesis 3 – Step-Level Funnel Conversion (Drop-Off Analysis)

The overall completion rate shows whether clients reach the final `confirm` step, but it does not show **where clients drop off in the funnel**.

To identify where the redesigned interface reduces or increases drop-off, we compare the probability of progressing to the next step for Test and Control at each funnel transition:

- `start → step_1`
- `step_1 → step_2`
- `step_2 → step_3`
- `step_3 → confirm`

### Hypotheses

For each transition:

**Null hypothesis (H₀):**

\[
H_0: p_{Test} = p_{Control}
\]

The probability of progressing to the next step is the same for Test and Control.

**Alternative hypothesis (H₁):**

\[
H_1: p_{Test} \neq p_{Control}
\]

The probability of progressing to the next step differs between Test and Control.

Because four separate transitions are tested, a **Bonferroni correction** is applied to control the overall probability of a Type I error.

With an overall significance level of:

\[
\alpha = 0.05
\]

the adjusted significance level for each test is:

\[
\alpha_{adjusted} = \frac{0.05}{4} = 0.0125
\]

A transition is therefore considered statistically significant only when its p-value is below **0.0125**.

In [54]:
# Prepare step-level conversion rates for hypothesis testing

step_order = ["start", "step_1", "step_2", "step_3", "confirm"]

transition_rows = []

for variation in ["Control", "Test"]:
    
    group = (
        funnel_summary[
            funnel_summary["Variation"] == variation
        ]
        .set_index("process_step")
    )

    for i in range(len(step_order) - 1):
        
        from_step = step_order[i]
        to_step = step_order[i + 1]

        clients_from = int(group.loc[from_step, "clients_reached"])
        clients_to = int(group.loc[to_step, "clients_reached"])

        transition_rows.append({
            "Variation": variation,
            "transition": f"{from_step} → {to_step}",
            "clients_from": clients_from,
            "clients_to": clients_to,
            "conversion_rate": clients_to / clients_from,
            "drop_off_rate": 1 - (clients_to / clients_from)
        })

transition_df = pd.DataFrame(transition_rows)

transition_df

,Variation,transition,clients_from,clients_to,conversion_rate,drop_off_rate
0,Control,start → step_1,23526,20241,0.860367,0.139633
1,Control,step_1 → step_2,20241,18786,0.928116,0.071884
2,Control,step_2 → step_3,18786,17521,0.932663,0.067337
3,Control,step_3 → confirm,17521,15428,0.880543,0.119457
4,Test,start → step_1,26961,24507,0.908980,0.091020
5,Test,step_1 → step_2,24507,22528,0.919248,0.080752
6,Test,step_2 → step_3,22528,21118,0.937411,0.062589
7,Test,step_3 → confirm,21118,18682,0.884648,0.115352


In [55]:
from statsmodels.stats.proportion import proportions_ztest

hypothesis_3_results = []

transitions = transition_df["transition"].unique()

for transition in transitions:

    control = transition_df[
        (transition_df["Variation"] == "Control") &
        (transition_df["transition"] == transition)
    ].iloc[0]

    test = transition_df[
        (transition_df["Variation"] == "Test") &
        (transition_df["transition"] == transition)
    ].iloc[0]

    counts = np.array([
        test["clients_to"],
        control["clients_to"]
    ])

    nobs = np.array([
        test["clients_from"],
        control["clients_from"]
    ])

    z_stat, p_value = proportions_ztest(
        counts,
        nobs,
        alternative="two-sided"
    )

    hypothesis_3_results.append({
        "transition": transition,
        "control_conversion": control["conversion_rate"],
        "test_conversion": test["conversion_rate"],
        "difference": test["conversion_rate"] - control["conversion_rate"],
        "z_statistic": z_stat,
        "p_value": p_value
    })

hypothesis_3_results = pd.DataFrame(hypothesis_3_results)

hypothesis_3_results

,transition,control_conversion,test_conversion,difference,z_statistic,p_value
0,start → step_1,0.860367,0.908980,0.048612,17.166186,4.756647e-66
1,step_1 → step_2,0.928116,0.919248,-0.008869,-3.507971,4.515378e-04
2,step_2 → step_3,0.932663,0.937411,0.004749,1.953070,5.081125e-02
3,step_3 → confirm,0.880543,0.884648,0.004105,1.248737,2.117615e-01


In [56]:
# Apply Bonferroni correction for four simultaneous tests

alpha = 0.05
n_tests = len(hypothesis_3_results)
bonferroni_alpha = alpha / n_tests

hypothesis_3_results["bonferroni_alpha"] = bonferroni_alpha

hypothesis_3_results["significant_after_bonferroni"] = (
    hypothesis_3_results["p_value"] < bonferroni_alpha
)

print(f"Original alpha: {alpha}")
print(f"Number of tests: {n_tests}")
print(f"Bonferroni-adjusted alpha: {bonferroni_alpha}")

hypothesis_3_results

Original alpha: 0.05
Number of tests: 4
Bonferroni-adjusted alpha: 0.0125


,transition,control_conversion,test_conversion,difference,z_statistic,p_value,bonferroni_alpha,significant_after_bonferroni
0,start → step_1,0.860367,0.908980,0.048612,17.166186,4.756647e-66,0.0125,True
1,step_1 → step_2,0.928116,0.919248,-0.008869,-3.507971,4.515378e-04,0.0125,True
2,step_2 → step_3,0.932663,0.937411,0.004749,1.953070,5.081125e-02,0.0125,False
3,step_3 → confirm,0.880543,0.884648,0.004105,1.248737,2.117615e-01,0.0125,False


### Hypothesis 3 Conclusion

After applying the Bonferroni correction, the adjusted significance level is **α = 0.0125**.

Two funnel transitions show statistically significant differences between Test and Control:

- **start → step_1:** Test conversion is **90.90%**, compared with **86.04%** for Control, an improvement of approximately **4.86 percentage points**. This difference is statistically significant.
- **step_1 → step_2:** Test conversion is **91.92%**, compared with **92.81%** for Control, a decrease of approximately **0.89 percentage points**. This difference is also statistically significant.

The differences for **step_2 → step_3** and **step_3 → confirm** are not statistically significant after the Bonferroni correction.

**Conclusion:** The redesign's effect is concentrated in the earlier stages of the funnel. It substantially reduces drop-off from `start → step_1`, but slightly increases drop-off from `step_1 → step_2`. This suggests that the redesign improves initial engagement while introducing some friction at the following step.

## EXTRA Hypothesis 4 – Demographic Balance: Client Age

As an additional hypothesis, we test whether the Test and Control groups differ significantly in average client age.

This is useful because age may influence digital behaviour and comfort with an online interface. A significant age imbalance could therefore potentially affect the interpretation of the A/B test results.

### Hypotheses

**Null hypothesis (H₀):**

\[
H_0: \mu_{Test} = \mu_{Control}
\]

The average client age is the same in the Test and Control groups.

**Alternative hypothesis (H₁):**

\[
H_1: \mu_{Test} \neq \mu_{Control}
\]

The average client age differs between the Test and Control groups.

A **two-sided Welch independent-samples t-test** is used because the groups are independent and equal population variances are not assumed.

Significance level:

\[
\alpha = 0.05
\]

In [6]:
# Compare client age between Test and Control groups

test_age = experiment_final_df.loc[
    experiment_final_df["Variation"] == "Test",
    "clnt_age"
].dropna()

control_age = experiment_final_df.loc[
    experiment_final_df["Variation"] == "Control",
    "clnt_age"
].dropna()

print("Test group")
print(f"N: {len(test_age)}")
print(f"Mean age: {test_age.mean():.2f}")
print(f"Median age: {test_age.median():.2f}")

print("\nControl group")
print(f"N: {len(control_age)}")
print(f"Mean age: {control_age.mean():.2f}")
print(f"Median age: {control_age.median():.2f}")

NameError: name 'experiment_final_df' is not defined

In [5]:
from scipy.stats import ttest_ind

# Welch independent-samples t-test

t_stat_age, p_value_age = ttest_ind(
    test_age,
    control_age,
    equal_var=False
)

print(f"T-statistic: {t_stat_age:.4f}")
print(f"P-value: {p_value_age:.6f}")

if p_value_age < 0.05:
    print("Reject H0: average age differs significantly between Test and Control.")
else:
    print("Fail to reject H0: no statistically significant age difference was detected.")

NameError: name 'test_age' is not defined

### Hypothesis 4 Conclusion

The Test group had a mean age of **47.16 years**, compared with **47.50 years** for the Control group.

The Welch independent-samples t-test produced:

- **t-statistic = -2.4161**
- **p-value = 0.0157**

Since the p-value is below the significance level of 0.05, we **reject the null hypothesis**.

There is statistically significant evidence that the average age differs between the Test and Control groups.

However, the observed difference is only about **0.34 years**, which is very small in practical terms.

**Conclusion:** The groups are not perfectly balanced on age statistically, but the difference is minor and is unlikely by itself to explain the much larger differences observed in completion behaviour.